# scProteomicsBench
## Reproducible ML Benchmarking for Single-Cell Proteomics

**Dataset:** Real SCoPE2 data (Bioconductor `scpdata`)  
**Task:** Macrophage vs Monocyte classification  
**Best result:** LightGBM — **98.32% test accuracy**  

**Author:** Sepideh Moafi 
**Year:** 2025-2026
---


## 1. Setup & Installation

In [ ]:
!pip install -q xgboost lightgbm scikit-learn pandas numpy matplotlib seaborn pytest

## 2. Load Real SCoPE2 Data

Using the official Bioconductor `scpdata` package.  
Original publication: Specht et al. 2019, *Single-cell proteomic and transcriptomic analysis of macrophage heterogeneity using SCoPE2*.


In [ ]:
import subprocess
code = r'''
library(scpdata)
library(QFeatures)
scp <- specht2019v3()
write.csv(as.data.frame(assay(scp, "proteins")), "/content/scope2_proteins.csv")
write.csv(as.data.frame(colData(scp)), "/content/scope2_cell_metadata.csv")
'''
subprocess.run(["R", "--vanilla", "-e", code])
print("✅ Data loaded")

## 3. Prepare Dataset

In [ ]:
import pandas as pd
import numpy as np
import os

proteins_df = pd.read_csv("/content/scope2_proteins.csv", index_col=0)
meta_df = pd.read_csv("/content/scope2_cell_metadata.csv", index_col=0)

X_df = proteins_df.T
common = X_df.index.intersection(meta_df.index)
X_df = X_df.loc[common]
meta_df = meta_df.loc[common]

mask = meta_df["SampleType"].isin(["Macrophage", "Monocyte"])
X_final = X_df.loc[mask]
y_final = meta_df.loc[mask, "SampleType"].values

os.makedirs("/content/scProteomicsBench/data/processed", exist_ok=True)
df_out = X_final.copy()
df_out["cell_type"] = y_final
df_out.to_csv("/content/scProteomicsBench/data/processed/scope2_real.csv", index=False)

print(f"✅ Dataset: {df_out.shape}")
print(f"📊 Classes: {df_out['cell_type'].value_counts().to_dict()}")

## 4. Run Benchmark (4 models)

In [ ]:
%cd /content/scProteomicsBench
!python benchmarks/run_benchmark.py data/processed/scope2_real.csv

## 5. Results Summary

In [ ]:
import json
import pandas as pd

with open("results/benchmark_results.json") as f:
    results = json.load(f)

rows = []
for name, r in results.items():
    if "error" in r: continue
    rows.append({
        "Model": name,
        "CV F1": f"{r['cv_f1_macro_mean']:.4f} ± {r['cv_f1_macro_std']:.4f}",
        "Test Accuracy": f"{r['test_metrics']['accuracy']:.4f}",
        "F1 Macro": f"{r['test_metrics']['f1_macro']:.4f}",
        "Time (s)": f"{r['train_time_sec']:.2f}",
    })

df = pd.DataFrame(rows)
print("=" * 80)
print("📊 BENCHMARK RESULTS — Real SCoPE2 Data")
print("=" * 80)
print(df.to_string(index=False))
print("=" * 80)

## 6. Performance Visualization

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np

with open("results/benchmark_results.json") as f:
    results = json.load(f)
results = {k: v for k, v in results.items() if "error" not in v}

models = list(results.keys())
acc = [results[m]["test_metrics"]["accuracy"] for m in models]
f1 = [results[m]["test_metrics"]["f1_macro"] for m in models]
cv = [results[m]["cv_f1_macro_mean"] for m in models]
cv_std = [results[m]["cv_f1_macro_std"] for m in models]

colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Accuracy
bars1 = axes[0].bar(models, acc, color=colors, edgecolor='black')
for b, a in zip(bars1, acc):
    axes[0].text(b.get_x()+b.get_width()/2, b.get_height()+0.003,
                 f'{a:.3f}', ha='center', fontweight='bold')
axes[0].set_ylabel('Test Accuracy')
axes[0].set_title('Test Accuracy on Real SCoPE2 Data', fontweight='bold')
axes[0].set_ylim(0.9, 1.0)
axes[0].tick_params(axis='x', rotation=15)

# CV F1
bars2 = axes[1].bar(models, cv, yerr=cv_std, capsize=8, color=colors, edgecolor='black')
for b, c in zip(bars2, cv):
    axes[1].text(b.get_x()+b.get_width()/2, b.get_height()+0.003,
                 f'{c:.3f}', ha='center', fontweight='bold')
axes[1].set_ylabel('CV F1 Macro')
axes[1].set_title('Cross-Validation F1 (3-fold)', fontweight='bold')
axes[1].set_ylim(0.9, 1.0)
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

## 7. Confusion Matrices

In [ ]:
import json
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

with open("results/benchmark_results.json") as f:
    results = json.load(f)
results = {k: v for k, v in results.items() if "error" not in v}

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for i, model in enumerate(results.keys()):
    cm = np.array(results[model]["test_metrics"]["confusion_matrix"])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                cbar=False, xticklabels=['Macrophage', 'Monocyte'],
                yticklabels=['Macrophage', 'Monocyte'],
                annot_kws={"size": 14, "weight": "bold"})
    acc = results[model]["test_metrics"]["accuracy"]
    axes[i].set_title(f'{model}\nAccuracy: {acc:.3f}', fontweight='bold')
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')

plt.suptitle('Confusion Matrices — Test Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Conclusions

### Key Findings
- **Best model:** LightGBM — **98.32%** test accuracy
- **Fastest model:** LogisticRegression — 0.40s
- **All models:** >94% accuracy (real biological signal)
- **Dataset:** 3,042 proteins × 1,490 cells from real SCoPE2 data

### Reproducibility
- ✅ Modular Python architecture (`src/`)
- ✅ Leakage-safe preprocessing (CLR + KNN imputation)
- ✅ Cell-aware StratifiedKFold cross-validation
- ✅ Pytest test suite
- ✅ GitHub Actions CI
- ✅ Dockerfile

### Technologies
`Python` · `scikit-learn` · `XGBoost` · `LightGBM` · `pandas` · `NumPy` · `pytest` · `GitHub Actions` · `Docker`
